# SHIPP — Validate Silver Coordinates

**Review draft — read-only.** This notebook does not write to Bronze, Silver, Gold, or Lakebase.

## Goal
Prove that the coordinates persisted in:
- `bootcamp_students.shipp_silver.silver_listings`
- `bootcamp_students.shipp_silver.silver_requests`

match the **newest non-deleted Bronze CDC event** for the same business key.

This is stronger than checking `LIMIT 1`: it validates the entire current-state coordinate lineage.

## PASS criteria
- Silver tables exist.
- No duplicate business keys.
- No impossible latitude/longitude values.
- Silver and current Bronze have the same key set.
- `latitude` and `longitude` exactly match the newest Bronze event.
- Silver source trace columns point to the same Bronze event.

In [ ]:
%run ../common/shipp_silver_lib

In [ ]:
REQUIRED = [BRONZE_LISTINGS_HISTORY, BRONZE_REQUESTS_HISTORY, SILVER_LISTINGS, SILVER_REQUESTS]
missing = [t for t in REQUIRED if not table_exists(t)]
print("Required tables:")
for t in REQUIRED:
    print(" ", "OK " if table_exists(t) else "MISSING", t)
assert not missing, f"BLOCKED: missing required table(s): {missing}"

In [ ]:
def validate_coordinate_lineage(bronze_table: str, silver_table: str, key: str):
    bronze_df, bronze_version = read_bronze_pinned(bronze_table)
    _, current_rows_df, deleted_keys_df = reconstruct_current_state(bronze_df, key)
    silver_df = spark.table(silver_table)

    # Key-set and traceability checks
    duplicates = duplicate_key_count(silver_df, key)
    missing_current = current_rows_df.select(key).join(silver_df.select(key), key, "left_anti").count()
    extra_silver = silver_df.select(key).join(current_rows_df.select(key), key, "left_anti").count()
    resurrected = silver_df.select(key).join(deleted_keys_df, key, "inner").count()

    joined = (
        silver_df.alias("s")
        .join(current_rows_df.alias("b"), key, "inner")
    )

    coordinate_mismatches = joined.filter(
        (~F.col("s.latitude").eqNullSafe(F.col("b.latitude")))
        | (~F.col("s.longitude").eqNullSafe(F.col("b.longitude")))
    )

    trace_mismatches = joined.filter(
        (~F.col("s.source_pg_lsn").eqNullSafe(F.col("b._pg_lsn")))
        | (~F.col("s.source_sort_by").eqNullSafe(F.col("b._sort_by")))
        | (~F.col("s.source_change_type").eqNullSafe(F.col("b._pg_change_type")))
    )

    out_of_range = out_of_range_coordinate_count(silver_df)

    print(f"\n=== {silver_table} ===")
    print("Bronze version:                    ", bronze_version)
    print("Silver rows:                       ", silver_df.count())
    print("Duplicate business keys:           ", duplicates)
    print("Missing current Bronze keys:       ", missing_current)
    print("Extra Silver keys:                 ", extra_silver)
    print("Resurrected deleted keys:          ", resurrected)
    print("Coordinate mismatches:             ", coordinate_mismatches.count())
    print("Source-event trace mismatches:     ", trace_mismatches.count())
    print("Out-of-range coordinate rows:      ", out_of_range)

    if coordinate_mismatches.count():
        display(
            coordinate_mismatches.select(
                F.col(f"s.{key}").alias(key),
                F.col("b.latitude").alias("bronze_latitude"),
                F.col("b.longitude").alias("bronze_longitude"),
                F.col("s.latitude").alias("silver_latitude"),
                F.col("s.longitude").alias("silver_longitude"),
                F.col("b._pg_lsn").alias("bronze_pg_lsn"),
                F.col("s.source_pg_lsn").alias("silver_source_pg_lsn"),
            ).orderBy(key)
        )

    run_quality_gate({
        "duplicate business key": duplicates,
        "missing current Bronze key": missing_current,
        "extra Silver key": extra_silver,
        "resurrected deleted key": resurrected,
        "coordinate mismatch": coordinate_mismatches.count(),
        "source-event trace mismatch": trace_mismatches.count(),
        "out-of-range coordinate": out_of_range,
    }, row_count=silver_df.count(), allow_empty=False)

    return {
        "table": silver_table,
        "rows": silver_df.count(),
        "bronze_version": bronze_version,
        "coordinate_mismatches": coordinate_mismatches.count(),
        "status": "PASS",
    }

In [ ]:
listing_result = validate_coordinate_lineage(
    BRONZE_LISTINGS_HISTORY,
    SILVER_LISTINGS,
    "listing_id",
)

request_result = validate_coordinate_lineage(
    BRONZE_REQUESTS_HISTORY,
    SILVER_REQUESTS,
    "request_id",
)

In [ ]:
print("\nRecent routable Silver rows:")
display(
    spark.table(SILVER_LISTINGS)
    .filter(F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
    .select("listing_id", "latitude", "longitude", "source_pg_lsn", "source_change_type", "updated_at")
    .orderBy(F.col("updated_at").desc_nulls_last())
    .limit(20)
)

display(
    spark.table(SILVER_REQUESTS)
    .filter(F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
    .select("request_id", "latitude", "longitude", "source_pg_lsn", "source_change_type", "updated_at")
    .orderBy(F.col("updated_at").desc_nulls_last())
    .limit(20)
)

In [ ]:
summary = {
    "stage": "silver_coordinate_validation",
    "listings": listing_result,
    "requests": request_result,
    "status": "PASS",
}
print(json.dumps(summary, indent=2, default=str))